*GPU Programming - Aljoscha Rheinwalt <aljoscha.rheinwalt@uni-potsdam.de>*

---

# CUDA GPU Programming in Python using numba

## Convolutions, Shading, Ray Tracing, Shadow Casting

---

This lab is designed to run on a host with a GPU. For debugging purposes, the same code can also be run on the CPU by enabling CUDA simulation:

In [ ]:
# import os
# os.environ["NUMBA_ENABLE_CUDASIM"] = "1"
# os.environ["NUMBA_CUDA_DEBUGINFO"] = "1"

Alternatively, you can run the notebook on a remote host with a GPU and access it locally by forwarding JupyterLab’s default port, 8888, through an SSH tunnel:
```
ssh -L 8888:localhost:8888 remote_host
```

```
    $ python3 -m venv env
    $ source env/bin/activate
    (env) $ pip install numba-cuda
    (env) $ pip install gdal
    ...
```

In [ ]:
# optional: (location, time) -> sun position
import pvlib

In [ ]:
import numpy as np
from time import time
from osgeo import gdal

from matplotlib import pyplot as pl
pl.rc("figure", dpi=150)

In [ ]:
from numba import cuda

cuda.detect()

         CPU side                                             GPU side
         ========                                             ========

    +------------------+                               +----------------------+
    |                  |                               |                      |
    |      CPU         |                               |     GPU cores        |
    |  few strong      |                               |  many small workers  |
    |  workers         |                               |                      |
    +--------+---------+                               +----------+-----------+
             |                                                    |
             |                                                    |
    +--------v---------+        PCIe / system bus      +----------v-----------+
    |                  |   =========================>  |                      |
    |     CPU RAM      |        narrow bridge          |      GPU RAM         |
    |  large, general  |   <=========================  |  fast, local to GPU  |
    |                  |        data transfer          |                      |
    +------------------+                               +----------------------+

## CPU thinking

---

    for y in range(height):
        for x in range(width):
            process pixel(x, y)


## GPU thinking

---

Start thousands of threads at once and each thread gets one pixel.

Grid of blocks (blocks per grid):

     +----------+----------+----------+
     | block 0  | block 1  | block 2  |
     +----------+----------+----------+
     | block 3  | block 4  | block 5  |
     +----------+----------+----------+

Inside each block (thread per block):

     +----+----+----+----+
     | t0 | t1 | t2 | t3 |
     +----+----+----+----+
     | t4 | t5 | t6 | t7 |
     +----+----+----+----+
     | t8 | t9 | t10| t11|
     +----+----+----+----+
     
Global thread index:

    global_x = blockIdx.x * blockDim.x + threadIdx.x
    global_y = blockIdx.y * blockDim.y + threadIdx.y

## Large raster data

---

In [ ]:
img = pl.imread("Ravi1_Orthomosaic_0.2mm.jpg")
pl.figure(1, (15, 8))
pl.imshow(img)
pl.show()

In [ ]:
img.shape

In [ ]:
def grayscale(rgb):
    return rgb[:, :, 0] * 0.2989 + rgb[:, :, 1] * 0.5870 + rgb[:, :, 2] * 0.1140
    

In [ ]:
%%time

gray = grayscale(img)

In [ ]:
pl.figure(1, (15, 8))
pl.imshow(gray, cmap="gray")
pl.show()

In [ ]:
@cuda.jit
def cuda_grayscale(out, rgb, n, m):
    i, j = cuda.grid(2)
    if i < n and j < m:
        out[i, j] = rgb[i, j, 0] * 0.2989 + rgb[i, j, 1] * 0.5870 + rgb[i, j, 2] * 0.1140
        

In [ ]:
%%time

# move data to GPU
d_out = cuda.device_array(img.shape[:2], np.float32)
d_img = cuda.to_device(img)

# thread grid layout
nthreads = (32, 32)
nblocksy = (img.shape[0] + nthreads[0] - 1) // nthreads[0]
nblocksx = (img.shape[1] + nthreads[1] - 1) // nthreads[1]
nblocks = (nblocksy, nblocksx)

# GPU kernel call
cuda_grayscale[nblocks, nthreads](d_out, d_img, img.shape[0], img.shape[1])

# copy data from GPU to host
out = d_out.copy_to_host()

In [ ]:
pl.figure(1, (15, 8))
pl.imshow(out - gray, cmap="seismic")
cb = pl.colorbar(location="top", aspect=100)
cb.set_label("Gray scale difference")
pl.show()

In [ ]:
# move data to GPU
t = time()
d_out = cuda.device_array(img.shape[:2], np.float32)
t1 = (time() - t)*1000
print("d_out = cuda.device_array(): %.1f ms" % t1)

t = time()
d_img = cuda.to_device(img)
t2 = (time() - t)*1000
print("d_img = cuda.to_device(): %.1f ms" % t2)

# thread grid layout
nthreads = (32, 32)
nblocksy = (img.shape[0] + nthreads[0] - 1) // nthreads[0]
nblocksx = (img.shape[1] + nthreads[1] - 1) // nthreads[1]
nblocks = (nblocksy, nblocksx)

# GPU kernel call
t = time()
cuda_grayscale[nblocks, nthreads](d_out, d_img, img.shape[0], img.shape[1])
t3 = (time() - t)*1000
print("cuda_grayscale[](): %.1f ms" % t3)

t = time()
cuda.synchronize()
t4 = (time() - t)*1000
print("cuda.synchronize(): %.1f ms" % t4)

# copy data from GPU to host
t = time()
out = d_out.copy_to_host()
t5 = (time() - t)*1000
print("out = d_out.copy_to_host(): %.1f ms" % t5)

print("sum: %.1f ms" % (t1+t2+t3+t4+t5))

In [ ]:
img.dtype

## Gaussian convolutional filter

---

In contrast to Dask we have a global shared memory model with CUDA. We can have arbitrarily non-local operations without the effort of overlap management. In the following an example of a convolutional filter.

In [ ]:
@cuda.jit("float32[:,:], float32[:,:], float32[:,:], int64, int64")
def cuda_conv_filter(out, inp, w, n, m):
    i, j = cuda.grid(2)
    if 3 <= i and i < n - 3 and 3 <= j and j < m - 3:
        out[i, j] = 0
        for ii in range(5):
            for jj in range(5):
                out[i, j] += w[ii, jj] * inp[i-2+ii, j-2+jj]
    

In [ ]:
def gaussian_kernel_weights():
    from math import exp
    w01 = exp(-0.5)
    w02 = exp(-2.0)
    w11 = exp(-1)
    w12 = exp(-2.5)
    w22 = exp(-4)
    w = np.array([
            [w22, w12, w02, w12, w22],
            [w12, w11, w01, w11, w12],
            [w02, w01,   1, w01, w02],
            [w12, w11, w01, w11, w12],
            [w22, w12, w02, w12, w22]
    ])
    w /= w.sum()
    return w


In [ ]:
# move data to GPU
t = time()
d_out = cuda.device_array(gray.shape, np.float32)
t1 = (time() - t)*1000
print("d_out = cuda.device_array(): %.1f ms" % t1)

t = time()
d_inp = cuda.to_device(gray.astype("float32"))
t2 = (time() - t)*1000
print("d_inp = cuda.to_device(): %.1f ms" % t2)

weights = gaussian_kernel_weights()
t = time()
d_w = cuda.to_device(weights.astype("float32"))
t2 = (time() - t)*1000
print("d_w = cuda.to_device(): %.1f ms" % t2)

# thread grid layout
nthreads = (32, 32)
nblocksy = (img.shape[0] + nthreads[0] - 1) // nthreads[0]
nblocksx = (img.shape[1] + nthreads[1] - 1) // nthreads[1]
nblocks = (nblocksy, nblocksx)

# GPU kernel call
t = time()
cuda_conv_filter[nblocks, nthreads](d_out, d_inp, d_w, d_inp.shape[0], d_inp.shape[1])
t3 = (time() - t)*1000
print("cuda_conv_filter[](): %.1f ms" % t3)

t = time()
cuda.synchronize()
t4 = (time() - t)*1000
print("cuda.synchronize(): %.1f ms" % t4)

# copy data from GPU to host
t = time()
out = d_out.copy_to_host()
t5 = (time() - t)*1000
print("out = d_out.copy_to_host(): %.1f ms" % t5)

print("sum: %.1f ms" % (t1+t2+t3+t4+t5))

In [ ]:
fg, ax = pl.subplots(1, 2, figsize=(15, 8))
ax[0].imshow(gray[:150, :150], cmap="gray")
ax[1].imshow(out[:150, :150], cmap="gray")
ax[0].axis("off")
ax[1].axis("off")
pl.tight_layout()
pl.show()

In [ ]:
from scipy.ndimage import gaussian_filter


def gaussian_filter_gpu(inp):
    weights = gaussian_kernel_weights()
    d_wgh = cuda.to_device(weights.astype("float32"))
    d_inp = cuda.to_device(inp.astype("float32"))
    d_out = cuda.device_array(inp.shape, np.float32)
    nthreads = (32, 32)
    nblocksy = (img.shape[0] + nthreads[0] - 1) // nthreads[0]
    nblocksx = (img.shape[1] + nthreads[1] - 1) // nthreads[1]
    nblocks = (nblocksy, nblocksx)
    cuda_conv_filter[nblocks, nthreads](d_out, d_inp, d_wgh, d_inp.shape[0], d_inp.shape[1])
    out = d_out.copy_to_host()
    return out
    

In [ ]:
%%time

f = gaussian_filter(gray, sigma=1)

In [ ]:
%%time

g = gaussian_filter_gpu(gray)

CuPy is generally a little easier to use and has many Numpy and SciPy functions readily available as GPU versions:

In [ ]:
import cupy as cp
from cupyx.scipy.ndimage import gaussian_filter as gaussian_filter_cupy

In [ ]:
%%time

h = gaussian_filter_cupy(cp.asarray(gray), sigma=1).get()

In [ ]:
%%time

gray_cupy = cp.asarray(gray)

In [ ]:
%%time

h_cupy = gaussian_filter_cupy(gray_cupy, sigma=1)

In [ ]:
%%time

h = h_cupy.get()

In [ ]:
fg, ax = pl.subplots(1, 3, figsize=(15, 8))
ax[0].imshow(f[1000:2500,1000:2000], cmap="gray")
ax[1].imshow(g[1000:2500,1000:2000], cmap="gray")
ax[2].imshow(h[1000:2500,1000:2000], cmap="gray")
ax[0].axis("off")
ax[1].axis("off")
ax[2].axis("off")
pl.tight_layout()
pl.show()

## Raking light

---

*Raking light is the illumination of objects from a light source at an oblique angle or almost parallel to the surface. This type of illumination provides information on the surface topography and relief of the artefact thus lit. It is widely used in the examination of works of art.* ([wikipedia](https://en.wikipedia.org/wiki/Raking_light))

![raking light](raking_light.jpg)

[Raking light](https://en.wikipedia.org/wiki/Raking_light),
[Shading](https://en.wikipedia.org/wiki/Shading), 
[Shadow mapping](https://en.wikipedia.org/wiki/Shadow_mapping), 
[Shadow volume](https://en.wikipedia.org/wiki/Shadow_volume).

---

In [ ]:
# DSM geotiff
fn_dsm = "COP_10m_PunaECordillera_int16_epsg32719.tif"

gdal.DontUseExceptions()

with gdal.Open(fn_dsm) as fp:
    dsm = fp.ReadAsArray()

dsm = dsm.astype("float")
dsm = np.flipud(dsm)
print(dsm.shape)
dsm[dsm == 0] = np.nan
dsm = dsm[10000:13000, 7500:11500]
print(dsm.shape)

fg, ax = pl.subplots(1, 1, figsize=(15, 8))
im = ax.imshow(dsm, interpolation="none", origin="lower", cmap="pink")
cb = fg.colorbar(im, ax=ax, orientation="horizontal", aspect=100)
cb.set_label("Elevation [m]")
pl.show()

### Gradients and surface normals

In higher dimensions the concept of a derivative translates to the
gradient $\nabla f$ of a scalar function $f$:
$$ \nabla f = \left(\frac{\partial f}{\partial x},
\frac{\partial f}{\partial y},
\frac{\partial f}{\partial z}\right) \;.$$
In 3D such a scalar function $f$ generally depends on $x, y$ and $z$
and if $f(x,y,z)$ defines a surface, its gradient is normal to that
surface.

For instance, given a sphere
$$ f(x, y, z) = x^2 + y^2 + z^2 - r^2 = 0\;,$$
the gradient is given by
$$ \nabla f = (2x, 2y, 2z) \;.$$

For a surface represented by a DSM the surface does not explicitly
depend on $z$, but can be parameterized by $x$ and $y$:
$$ z = g(x, y) \;.$$
This can be expressed as a surface $f(x,y,z)$ in the following way
$$ f(x,y,z) = z - g(x, y) = 0 \;,$$
so that the gradient is given by
$$ \nabla f = \left(\frac{\partial f}{\partial x},
\frac{\partial f}{\partial y}, 1 \right) =
\left(-\frac{\partial g}{\partial x},
-\frac{\partial g}{\partial y}, 1 \right) \;.$$

Surface normals $\vec{n}$ are then estimated as:

In [ ]:
def normals(dsm, pixelsize=1.0):
    dy, dx = np.gradient(dsm, pixelsize)
    dy, dx = dy.ravel(), dx.ravel()
    nn = np.c_[-dx, -dy, np.ones(dx.shape)]
    return (nn.T / np.sqrt(np.sum(nn * nn, axis=1)).T).T
    

If you use ENU coordinates with:

    x = East
    y = North
    z = Up

then a convenient convention for the sun direction is

$$ \vec{s} = (\cos⁡\theta \sin⁡\phi, \cos⁡\theta \cos⁡\phi, \sin⁡\theta)$$

where $\theta$ is the solar elevation above the horizon (0° horizon, 90° zenith) and $\phi$ is the azimuth measured clockwise from North:

    0° = North
    90° = East
    180° = South
    270° = West

Direct solar energy irradiance $E_{direct}$ is
$$E_{direct} \propto \cos\alpha = \vec{n}\cdot\vec{s} \;,$$
with $\alpha$ being the angle between the sun and surface direction.

In [ ]:
import pvlib

loc = pvlib.location.Location(
    latitude=51.34,
    longitude=12.37,
    tz='Europe/Berlin'
)

solpos = loc.get_solarposition(
    '2026-06-24 15:00:00'
)

print(solpos[['apparent_elevation', 'azimuth']])

In [ ]:
# normals
nn = normals(dsm, pixelsize=10.0)

# sun direction
theta = np.deg2rad(40)
phi = np.deg2rad(300)
sun = np.array([np.cos(theta) * np.sin(phi), np.cos(theta) * np.cos(phi), np.sin(theta)])

# shading as dot product of normals and sun direction
shading = np.dot(nn, sun)
shading[shading < 0] = 0
shading.shape = dsm.shape

In [ ]:
fg, ax = pl.subplots(1, 1, figsize=(15, 8))
im = ax.imshow(shading, interpolation="none", origin="lower", cmap="binary_r")
cb = fg.colorbar(im, ax=ax, orientation="horizontal", aspect=100)
cb.set_label(r"$n \cdot s$")
pl.show()

In [ ]:
fg, ax = pl.subplots(1, 2, figsize=(15, 8))
im = ax[0].imshow(dsm, interpolation="none", origin="lower", cmap="pink")
ax[1].imshow(dsm, interpolation="none", origin="lower", cmap="pink")
ax[1].imshow(shading, alpha=0.5, interpolation="none", origin="lower", cmap="binary_r")
cb = fg.colorbar(im, ax=ax, orientation="horizontal", aspect=100)
cb.set_label("Elevation [m]")
pl.show()

However, more specifically, the direct solar energy irradiance $E_{direct}$ is
$$E_{direct}(x, y, t) = V(x, y, t) E_{DNI}(t) \vec{n}\cdot\vec{s} \;,$$
with $V(x, y, t)$ being the binary visibility or shadow term.

---

![ray tracing](https://upload.wikimedia.org/wikipedia/commons/5/5c/Ray_trace_diagram.png)

![Sciography, the art of architectural shadows](https://upload.wikimedia.org/wikipedia/commons/thumb/c/cc/Tav20-rappresentazione.jpg/960px-Tav20-rappresentazione.jpg)

In [ ]:
def draw_line(y0, x0, phi_deg, shape):
    """
    cumpute pixel indices of line from (x0, y0) in the direction of phi
    """
    phi = np.deg2rad(phi_deg)
    dx = np.sin(phi)
    dy = np.cos(phi)

    # step size
    step = 1.0 / max(abs(dx), abs(dy))

    y_start, x_start = y0 + 0.5, x0 + 0.5

    # ray boundary intersection
    if dx > 0:
        tx = (shape[1] - x_start) / dx
    elif dx < 0:
        tx = (0 - x_start) / dx
    else:
        tx = np.inf

    if dy > 0:
        ty = (shape[0] - y_start) / dy
    elif dy < 0:
        ty = (0 - y_start) / dy
    else:
        ty = np.inf

    # ray length
    tmax = min(tx, ty)

    # number of steps
    n = int(np.floor(tmax / step)) + 1

    # sample line
    t = np.arange(n) * step
    x = np.floor(x_start + dx * t).astype(int)
    y = np.floor(y_start + dy * t).astype(int)

    return y, x, t

In [ ]:
# simple example line drawing
z = np.zeros((90, 160))
z[30:60, 40:80] = 20
z[50:70, 100:120] = 10

# draw a line from (y0, x0)
y0, x0 = 80, 20
y, x, t = draw_line(y0, x0, 135, z.shape)
z[y, x] = 15

pl.imshow(z, origin="lower", cmap="Purples")
pl.show()

In [ ]:
def cast_shadow(z, phi_deg, theta_deg, pixelsize=1.0):
    """
    ray shadow casting from each pixel (y0, x0) towards the sun
    """
    tan = np.tan(np.deg2rad(theta_deg))
    shadow_map = np.zeros(z.shape, dtype="bool")
    for y0 in range(z.shape[0]):
        for x0 in range(z.shape[1]):
            y, x, t = draw_line(y0, x0, phi_deg, z.shape)
            if len(x) == 0:
                continue
            r = t * pixelsize
            z_ray = z[y0, x0] + r * tan
            if np.any(z[y, x] > z_ray):
                shadow_map[y0, x0] = True
                
    return shadow_map.astype("short")

# simple example
z = np.zeros((90, 160))
z[30:60, 40:80] = 20
z[50:70, 100:120] = 10

# sun from southeast, morning-ish
s = cast_shadow(z, 135, 25)

pl.imshow(z, origin="lower", cmap="Purples")
pl.imshow(s, origin="lower", cmap="Greys", alpha=s/2)
pl.show()

In [ ]:
phi = np.arange(0, 360, 10)
theta = np.linspace(10, 80, len(phi))

shadows = [cast_shadow(z, p, t) for p, t in zip(phi, theta)]

In [ ]:
from matplotlib import animation
from IPython.display import HTML


def animate_shadow(z, shadows):
    fg, ax = pl.subplots(figsize=(15, 8))
    ax.imshow(z, origin="lower", cmap="Purples")

    shadow_img = ax.imshow(
        shadows[0],
        origin="lower",
        cmap="Greys",
        alpha=0.5 * shadows[0],
    )
    ax.set_axis_off()

    def animate(i):
        shadow = shadows[i]
        shadow_img.set_data(shadow)
        shadow_img.set_alpha(0.5 * shadow)
        return shadow_img

    anim = animation.FuncAnimation(
        fg,
        animate,
        frames=len(shadows),
        interval=200,
        blit=False,
    )
    pl.close(fg)
    return HTML(anim.to_jshtml())
    

In [ ]:
animate_shadow(z, shadows)

In [ ]:
def draw_dome(z, xg, yg, x0, y0, r0):
    """
    half spheres for a synthetic elevation model.
    """
    dx = xg - x0
    dy = yg - y0
    sl = np.where(dx*dx + dy*dy < r0*r0)
    dx, dy = dx[sl], dy[sl]
    dz = np.sqrt(r0*r0 - dx*dx - dy*dy)
    z[sl] = np.max((z[sl], dz), axis=0)
    return z
    

In [ ]:
xb = np.arange(320)
yb = np.arange(180)
xr = (xb[1:]+xb[:-1]) / 2
yr = (yb[1:]+yb[:-1]) / 2
xg, yg = np.meshgrid(xr, yr)
z = np.zeros((180, 320))
z[30:60, 40:80] = 20
z[50:70, 100:120] = 10
for i in range(10):
    x0 = np.random.random() * 320
    y0 = np.random.random() * 180
    r0 = np.random.random() * 50
    draw_dome(z, xg, yg, x0, y0, r0)
    
s = cast_shadow(z, 135, 40)
pl.imshow(z, origin="lower", cmap="Purples")
pl.imshow(s, origin="lower", cmap="Greys", alpha=s)
pl.show()

In [ ]:
phi = np.arange(0, 360, 10)
theta = np.linspace(10, 80, len(phi))

shadows = [cast_shadow(z, p, t) for p, t in zip(phi, theta)]

In [ ]:
animate_shadow(z, shadows)

In [ ]:
for i in range(len(shadows)):
    fg, ax = pl.subplots(figsize=(19.2, 10.8))
    ax.imshow(z, origin="lower", cmap="Purples")

    shadow_img = ax.imshow(
        shadows[i],
        origin="lower",
        cmap="Greys",
        alpha=0.5 * shadows[i],
    )
    ax.set_axis_off()
    pl.tight_layout()
    pl.savefig("img%03d.png" % i, dpi=100)
    pl.close(fg)
    

In [ ]:
from math import floor


@cuda.jit
def cuda_kern_cast_shadow(shadow, z, ys, xs, pixelsize, tan_theta, dy_sun, dx_sun, eps):
    y0, x0 = cuda.grid(2)
    if y0 >= ys or x0 >= xs:
        return
    
    # dominant-axis stepping
    adx = abs(dx_sun)
    ady = abs(dy_sun)
    if adx > ady:
        step = 1.0 / adx
    else:
        step = 1.0 / ady

    x_start = x0 + 0.5
    y_start = y0 + 0.5

    # ray / boundary
    if dx_sun > 0.0:
        tx = (xs - x_start) / dx_sun
    elif dx_sun < 0.0:
        tx = (0. - x_start) / dx_sun
    else:
        tx = 1.0e30

    if dy_sun > 0.0:
        ty = (ys - y_start) / dy_sun
    elif dy_sun < 0.0:
        ty = (0. - y_start) / dy_sun
    else:
        ty = 1.0e30

    # tmax = min(tx, ty)
    if tx < ty:
        tmax = tx
    else:
        tmax = ty

    # make sure to stay inside
    tmax -= 1.0e-6

    # number of samples along the ray
    n = int(tmax / step) + 1

    z0 = z[y0, x0]

    # start at 1 to skip the source pixel
    for k in range(1, n):
        t = k * step
        x = int(floor(x_start + dx_sun * t))
        y = int(floor(y_start + dy_sun * t))
        r = t * pixelsize
        dz = z[y, x] - z0 - r * tan_theta
        if dz > eps:
            shadow[y0, x0] = 1
            return

    shadow[y0, x0] = 0


def gpu_cast_shadow(z, phi_deg, theta_deg, pixelsize=1.0, eps=0):
    tan = np.tan(np.deg2rad(theta_deg))
    phi = np.deg2rad(phi_deg)
    dx = np.sin(phi)
    dy = np.cos(phi)
    ys, xs = z.shape
    d_z = cuda.to_device(z.astype("float32"))
    d_s = cuda.device_array((ys, xs), np.int16)
    
    nthreads = (32, 32)
    nblocksy = (ys + nthreads[0] - 1) // nthreads[0]
    nblocksx = (xs + nthreads[1] - 1) // nthreads[1]
    nblocks = (nblocksy, nblocksx)
    
    cuda_kern_cast_shadow[nblocks, nthreads](d_s, d_z, ys, xs, pixelsize, tan, dy, dx, eps)
    s = d_s.copy_to_host()
    return s
    

In [ ]:
%%time

s = cast_shadow(z, 60, 10)

In [ ]:
%%time

s = gpu_cast_shadow(z, 60, 10)

In [ ]:
# normals
nn = normals(dsm, pixelsize=10.0)

# sun direction
theta = np.deg2rad(10)
phi = np.deg2rad(60)
sun = np.array([np.cos(theta) * np.sin(phi), np.cos(theta) * np.cos(phi), np.sin(theta)])

# shading as dot product of normals and sun direction
shading = np.dot(nn, sun)
shading[shading < 0] = 0
shading.shape = dsm.shape

# shadows
shadows = gpu_cast_shadow(dsm, 60, 10, pixelsize=10.0)

fg, ax = pl.subplots(1, 2, figsize=(15, 8))
im = ax[0].imshow(dsm, interpolation="none", origin="lower", cmap="pink")
ax[0].imshow(shading, alpha=0.5, interpolation="none", origin="lower", cmap="binary_r")
ax[1].imshow(dsm, interpolation="none", origin="lower", cmap="pink")
ax[1].imshow(shading, alpha=0.5, interpolation="none", origin="lower", cmap="binary_r")
ax[1].imshow(shadows, alpha=shadows*0.9, interpolation="none", origin="lower", cmap="binary")
cb = fg.colorbar(im, ax=ax, orientation="horizontal", aspect=100)
cb.set_label("Elevation [m]")
pl.show()

In [ ]:
import pandas as pd

lat = -24.5812
lon = -66.6370
tz = "America/Argentina/Salta"

loc = pvlib.location.Location(
    latitude=lat,
    longitude=lon,
    tz=tz,
)

times = pd.date_range(
    "2026-06-24 00:00",
    "2026-06-24 23:59",
    freq="10min",
    tz=tz,
)

solpos = loc.get_solarposition(times)

phis = solpos["azimuth"].to_numpy()
thetas = solpos["apparent_elevation"].to_numpy()

daylight = thetas > 0

times = times[daylight]
phis = phis[daylight]
thetas = thetas[daylight]

print(phis)
print(thetas)

In [ ]:
pl.plot(times, thetas, "o")
pl.xlabel("t")
pl.ylabel(r"$\theta$")
pl.xticks(rotation=45)
pl.show()

In [ ]:
pl.plot(phis, thetas, "o")
pl.xlabel(r"$\phi$")
pl.ylabel(r"$\theta$")
pl.show()

In [ ]:
# normals
nn = normals(dsm, pixelsize=10.0)

for i in range(len(phis)):
    # sun direction
    theta = np.deg2rad(thetas[i])
    phi = np.deg2rad(phis[i])
    sun = np.array([np.cos(theta) * np.sin(phi), np.cos(theta) * np.cos(phi), np.sin(theta)])

    # shading as dot product of normals and sun direction
    shading = np.dot(nn, sun)
    shading[shading < 0] = 0
    shading.shape = dsm.shape

    # shadows
    shadows = gpu_cast_shadow(dsm, phis[i], thetas[i], pixelsize=10.0)

    fg, ax = pl.subplots(1, 1, figsize=(19.2, 10.8))
    im = ax.imshow(dsm, interpolation="none", origin="lower", cmap="pink")
    ax.imshow(shading, alpha=0.5, interpolation="none", origin="lower", cmap="binary_r")
    ax.imshow(shadows, alpha=shadows*0.9, interpolation="none", origin="lower", cmap="binary")
    cb = fg.colorbar(im, ax=ax, aspect=100)
    cb.set_label("Elevation [m]")
    pl.tight_layout()
    pl.savefig("img%03d.png" % i, dpi=100)
    pl.close(fg)
    